# EDA Auditado — TFG Guillermo Jiménez

Análisis Exploratorio de Datos auditado. Todos los estadísticos son fórmulas Excel que apuntan a datos reales.

**El Excel generado tendrá:**
- Hoja 0: `Datos_Brutos` — todos los jugadores con sus variables
- Hoja 1: `Estadísticos_UV` — fórmulas `=AVERAGE()`, `=MEDIAN()`, `=STDEV()`, etc.
- Hoja 2: `Correlaciones` — fórmula `=CORREL()` por cada par variable × log_VM
- Hoja 3: `PCA_Pasos` — matriz de covarianza y eigenvalores paso a paso
- Hoja 4: `Outliers` — filas filtradas con criterio IQR y z-score explícito
- Hoja 5: `Gráficos` — imágenes de matplotlib

In [6]:
!pip install -q openpyxl lxml seaborn matplotlib pandas numpy

## Imports y configuración

In [7]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings, re, unicodedata
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

warnings.filterwarnings('ignore')

## Rutas

In [8]:
import os
DB = '/content'

ENRICHED_V3 = os.path.join(DB, 'Dataset_Definitivo.xlsx')
OUT_DIR     = '/content/eda_plots'
OUT_EXCEL   = os.path.join(DB, 'EDA TFG v2.xlsx')
os.makedirs(OUT_DIR, exist_ok=True)

## Estilos Excel

In [9]:
H_FILL = PatternFill('solid', fgColor='1F4E79')
SH_FILL = PatternFill('solid', fgColor='2E75B6')
ST_FILL = PatternFill('solid', fgColor='DEEAF1')
WH_FILL = PatternFill('solid', fgColor='FFFFFF')
GR_FILL = PatternFill('solid', fgColor='E2EFDA')
OR_FILL = PatternFill('solid', fgColor='FFF2CC')
RD_FILL = PatternFill('solid', fgColor='FCE4D6')
H_FONT = Font(color='FFFFFF', bold=True, size=10)
B_FONT = Font(bold=True, size=9)
N_FONT = Font(size=9)
C_ALIGN = Alignment(horizontal='center', vertical='center')
L_ALIGN = Alignment(horizontal='left', vertical='center')
tside = Side(style='thin', color='B8CCE4')
BORDER = Border(left=tside, right=tside, top=tside, bottom=tside)

def hdr(ws, r, c, v, fill=H_FILL, font=H_FONT):
    cell = ws.cell(row=r, column=c, value=v)
    cell.fill, cell.font, cell.alignment, cell.border = fill, font, C_ALIGN, BORDER
    return cell

def val(ws, r, c, v, fmt=None, fill=WH_FILL, bold=False, align=C_ALIGN):
    cell = ws.cell(row=r, column=c, value=v)
    cell.fill = fill
    cell.font = B_FONT if bold else N_FONT
    cell.alignment = align
    cell.border = BORDER
    if fmt: cell.number_format = fmt
    return cell

def formula(ws, r, c, f, fmt=None, fill=WH_FILL, bold=False):
    cell = ws.cell(row=r, column=c, value=f)
    cell.fill = fill
    cell.font = B_FONT if bold else N_FONT
    cell.alignment = C_ALIGN
    cell.border = BORDER
    if fmt: cell.number_format = fmt
    return cell

## Carga de datos

In [10]:
SHEET_MAP = {
    'PL Players':'PL', 'LaLiga Players':'LaLiga',
    'Serie A Players':'SerieA', 'Bundesliga Players':'Bundesliga',
    'Ligue 1 Players':'Ligue1'
}

COL_ALIASES = {
    'Nombre':'name','Player':'name',
    'Posición':'position','Pos':'position',
    'Equipo':'team','Squad':'team',
    'Edad':'age_str','Age':'age_str',
    'Año de nacimiento':'birth_year','Born':'birth_year',
    'Partidos jugados esta temporada':'games','MP':'games',
    'Titularidades':'starts','Starts':'starts',
    'Minutos jugados esta temporada':'minutes','Min':'minutes',
    'Minutos totales/90':'min90','90s':'min90',
    'Goles':'goals_tm','Gls':'goals_tm',
    'Asistencias':'assists_tm','Ast':'assists_tm',
    'Amarillas':'yellows_tm','CrdY':'yellows_tm',
    'Rojas':'reds_tm','CrdR':'reds_tm',
    'Goles esperados':'xg','xG':'xg',
    'Asistencias esperadas':'xa','xAG':'xa',
    'Pases progresivos *':'prog_passes','PrgP':'prog_passes',
    'Recepciones de pases progresivos':'prog_rec','PrgR':'prog_rec',
    'Conducción de balón de al menos 10 metros hacia la portería rival':'prog_carries','PrgC':'prog_carries',
    'Valor de mercado':'market_value','market_value_eur':'market_value',
    'tm_player_id':'tm_id',
    'Años de contrato restantes':'contract_years',
    'contract_years_remaining':'contract_years',
    'Cantidad total de lesiones registradas':'total_injuries',
    'injury_count':'total_injuries',
    'Promedio de días de baja por lesión por temporada':'avg_inj_days',
    'injury_days_per_season':'avg_inj_days',
    'Número promedio de lesiones por temporada':'avg_inj_count',
    'injury_frequency':'avg_inj_count',
    'Ingreso total anual del club':'club_revenue',
    'club_revenue_eur':'club_revenue',
    'Categoría de ingresos del club**':'club_revenue_cat',
    'revenue_tier':'club_revenue_cat',
    '***Índice de madurez de carrera':'career_maturity',
    'Días de lesión en la temporada pasada':'inj_last_season',
    'inj_days_last1':'inj_last_season',
    'Media anual ponderada exponencialmente de los días de baja por lesión':'inj_ewa',
    'inj_ewa_days':'inj_ewa',
    'Riesgo de lesión':'inj_risk','inj_risk':'inj_risk',
    'Estado de forma':'inj_series_type',
    'inj_series_type':'inj_series_type',
    'Entradas ganadas (FBref)':'tackles_won',
    'Intercepciones (FBref)':'interceptions',
    'Faltas cometidas (FBref)':'fouls',
    'Faltas recibidas (FBref)':'fouled',
    'Centros (FBref)':'crosses',
    'Tarjetas amarillas (FBref)':'yellow_fbref',
    'Tarjetas rojas (FBref)':'red_fbref',
    'Goles (FBref)':'goals',
    'Tiros totales (FBref)':'shots',
    'Tiros a puerta (FBref)':'shots_on_target',
    'Precisión de tiro % (FBref)':'shot_accuracy',
    'Tiros por 90 min (FBref)':'shots_p90',
    'Penaltis marcados (FBref)':'pens_scored',
    'Penaltis tirados (FBref)':'pens_att',
    'PJ Portero (FBref)':'gk_games',
    'Titularidades Portero (FBref)':'gk_starts',
    'Goles encajados/90 (FBref)':'gk_ga90',
    'Tiros a puerta recibidos (FBref)':'gk_sot_against',
    'Paradas (FBref)':'gk_saves',
    '% Paradas (FBref)':'gk_save_pct',
    'Porterías a cero (FBref)':'gk_clean_sheets',
    '% Porterías a cero (FBref)':'gk_cs_pct',
}
def parse_age(v):
    if pd.isna(v): return np.nan
    m = re.match(r'(\d+)-(\d+)', str(v))
    if m: return int(m.group(1)) + int(m.group(2))/365
    try: return float(v)
    except: return np.nan

def clean_pos(p):
    if pd.isna(p): return 'Unknown'
    p = str(p).split(',')[0].split('/')[0].strip()
    if p in ('PO','GK'): return 'GK'
    if p in ('DF','CB','LB','RB','WB'): return 'DF'
    if p in ('MF','CM','DM','AM','LM','RM'): return 'MF'
    if p in ('FW','CF','LW','RW','ST','SS'): return 'FW'
    return p

def load_sheet(path, sheet, league):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    # Buscar la fila de encabezados: la con mas coincidencias en COL_ALIASES
    best_row = 0
    best_score = 0
    for i in range(min(5, len(raw))):
        row_vals = [str(h).strip() for h in raw.iloc[i] if not pd.isna(h)]
        score = sum(1 for v in row_vals if v in COL_ALIASES)
        if score > best_score:
            best_score = score
            best_row = i
    headers = []
    seen = {}
    for h in raw.iloc[best_row]:
        if pd.isna(h):
            mapped = None
        else:
            key = str(h).strip()
            mapped = COL_ALIASES.get(key, key if key else None)
        if mapped is not None:
            if mapped in seen:
                seen[mapped] += 1
                mapped = f'{mapped}_{seen[mapped]}'
            else:
                seen[mapped] = 0
        headers.append(mapped)
    df = raw.iloc[best_row + 1:].copy()
    df.columns = headers
    df = df.loc[:, [c for c in df.columns if c]]
    if 'name' not in df.columns:
        for col in df.columns:
            if df[col].dtype == object and df[col].notna().mean() > 0.5:
                sample = df[col].dropna().astype(str)
                if sample.str.contains(r'[A-Za-z]', regex=True).mean() > 0.8:
                    df = df.rename(columns={col: 'name'})
                    break
    if 'name' not in df.columns:
        print(f'AVISO: columna name no encontrada en hoja {sheet}. Columnas: {list(df.columns[:10])}')
        return pd.DataFrame()
    df = df[df['name'].notna() & (df['name'] != '')]
    df['league'] = league
    df['age'] = df['age_str'].apply(parse_age) if 'age_str' in df.columns else np.nan
    df['pos_clean'] = df['position'].apply(clean_pos) if 'position' in df.columns else 'Unknown'
    str_cols = {'name','position','team','league','age_str','club_revenue_cat','inj_risk','inj_series_type','tm_id','match_confidence','pos_clean'}
    for c in df.columns:
        if c not in str_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

print('Cargando datos...')
dfs = []
for sh, lg in SHEET_MAP.items():
    try:
        d = load_sheet(ENRICHED_V3, sh, lg)
        if len(d) > 0:
            dfs.append(d)
        else:
            print(f'  AVISO: hoja {sh} sin datos validos')
    except Exception as e:
        print(f'  ERROR en hoja {sh}: {e}')
if not dfs:
    raise ValueError('No se pudieron cargar datos de ninguna hoja')
df_all = pd.concat(dfs, ignore_index=True)
df_all['mv_eur'] = pd.to_numeric(df_all['market_value'], errors='coerce')
df_all['log_mv'] = np.where(df_all['mv_eur'] > 0, np.log(df_all['mv_eur']), np.nan)
df_valid = df_all[df_all['mv_eur'] > 0].copy()
print(f'  Total: {len(df_all)} jugadores | Con VM valido: {len(df_valid)}')

Cargando datos...
  Total: 1808 jugadores | Con VM valido: 1808


## Cálculo de estadísticos y correlaciones

In [11]:
ALL_FEATURES = [
    'age','min90','goals','shots','shots_on_target','shot_accuracy','shots_p90',
    'tackles_won','interceptions','fouls','fouled','crosses',
    'prog_carries','prog_passes','prog_rec',
    'contract_years','club_revenue','career_maturity',
    'total_injuries','avg_inj_days','avg_inj_count','inj_ewa','inj_last_season',
    'xg','xa',
    'gk_saves','gk_save_pct','gk_clean_sheets','gk_ga90',
]
FEATS = [f for f in ALL_FEATURES if f in df_valid.columns and df_valid[f].notna().mean() >= 0.10]

print('Calculando estadísticos...')

lmv = df_valid['log_mv'].dropna()
n_total = len(df_valid)

def pearson_manual(x, y):
    """r = Σ[(xi-x̄)(yi-ȳ)] / sqrt[Σ(xi-x̄)² · Σ(yi-ȳ)²]"""
    mask = x.notna() & y.notna()
    x, y = x[mask].values, y[mask].values
    if len(x) < 5: return np.nan, 0
    xm, ym = x.mean(), y.mean()
    num   = ((x - xm) * (y - ym)).sum()
    denom = np.sqrt(((x - xm)**2).sum() * ((y - ym)**2).sum())
    r = num / denom if denom > 0 else np.nan
    return round(float(r), 6), len(x)

corr_results = {}
for f in FEATS:
    r, n = pearson_manual(df_valid[f], df_valid['log_mv'])
    corr_results[f] = {'r': r, 'n': n, 'coverage': round(df_valid[f].notna().mean()*100, 1)}

corr_series = pd.Series({f: v['r'] for f, v in corr_results.items()}).dropna().sort_values(ascending=False)

Q1 = lmv.quantile(0.25)
Q3 = lmv.quantile(0.75)
IQR_val = Q3 - Q1
lo_iqr = Q1 - 1.5 * IQR_val
hi_iqr = Q3 + 1.5 * IQR_val
df_valid['z_score'] = (df_valid['log_mv'] - lmv.mean()) / lmv.std()
df_valid['outlier_iqr'] = (df_valid['log_mv'] < lo_iqr) | (df_valid['log_mv'] > hi_iqr)
df_valid['outlier_z'] = df_valid['z_score'].abs() > 3

PCA_FEATS = [f for f in [
    'age','min90','goals','shots','shots_p90',
    'tackles_won','interceptions','fouls','fouled','crosses',
    'prog_carries','prog_passes','contract_years','club_revenue',
    'total_injuries','avg_inj_days','inj_ewa',
] if f in df_valid.columns and df_valid[f].notna().mean() >= 0.30]

pca_df = df_valid[PCA_FEATS + ['log_mv','pos_clean','league']].dropna(subset=PCA_FEATS)
X = pca_df[PCA_FEATS].values.astype(float)
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0); X_std[X_std==0] = 1
X_sc   = (X - X_mean) / X_std
COV    = np.cov(X_sc.T)
eigvals, eigvecs = np.linalg.eigh(COV)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]; eigvecs = eigvecs[:, idx]
exp_var = eigvals / eigvals.sum()
cum_var = np.cumsum(exp_var)
PCs = X_sc @ eigvecs[:, :5]

print(f'  PCA: {len(pca_df)} jugadores × {len(PCA_FEATS)} features')
print(f'  Top r con log_mv: {corr_series.head(5).to_dict()}')

Calculando estadísticos...
  PCA: 1744 jugadores × 9 features
  Top r con log_mv: {'club_revenue': 0.539266, 'contract_years': 0.400249, 'prog_carries': 0.324225, 'prog_passes': 0.31149, 'prog_rec': 0.301422}


## Generación de gráficos

In [12]:
PALETTE = {'PL':'#3D195B','LaLiga':'#FF4B44','SerieA':'#1E5CA0','Bundesliga':'#D3010C','Ligue1':'#00305E'}
POS_C   = {'FW':'#E74C3C','MF':'#2ECC71','DF':'#3498DB','GK':'#F39C12'}
sns.set_theme(style='whitegrid')

print('Generando gráficos...')

# G1: Distribución MV
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución del Valor de Mercado', fontsize=13, fontweight='bold')
mv_m = df_valid['mv_eur'] / 1e6
axes[0].hist(mv_m, bins=60, color='#3498DB', edgecolor='white', lw=0.4, alpha=0.85)
axes[0].axvline(mv_m.median(), color='red', ls='--', lw=1.5, label=f'Mediana={mv_m.median():.1f}M€')
axes[0].set(xlabel='VM (€M)', ylabel='Frecuencia', title=f'Original  —  Skewness={mv_m.skew():.2f}')
axes[0].legend(fontsize=9)
axes[1].hist(lmv, bins=50, color='#2ECC71', edgecolor='white', lw=0.4, alpha=0.85)
axes[1].axvline(lmv.median(), color='red', ls='--', lw=1.5, label=f'Mediana={lmv.median():.2f}')
axes[1].set(xlabel='ln(VM €)', ylabel='', title=f'Log-normal  —  Skewness={lmv.skew():.2f}')
axes[1].legend(fontsize=9)
league_order = ['PL','LaLiga','SerieA','Bundesliga','Ligue1']
data_vio = [df_valid[df_valid['league']==l]['log_mv'].dropna().values for l in league_order]
data_vio = [d if len(d)>1 else np.array([0,1]) for d in data_vio]
parts = axes[2].violinplot(data_vio, positions=range(5), showmedians=True)
for i,pc in enumerate(parts['bodies']):
    pc.set_facecolor(list(PALETTE.values())[i]); pc.set_alpha(0.7)
axes[2].set(xticks=range(5), xticklabels=league_order, ylabel='ln(VM)', title='Por liga')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/G1_distribucion_mv.png', dpi=150, bbox_inches='tight'); plt.close()

# G2: Box por posición
pos_order = ['FW','MF','DF','GK']
fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot([df_valid[df_valid['pos_clean']==p]['log_mv'].dropna() for p in pos_order],
                labels=pos_order, patch_artist=True, medianprops=dict(color='black',lw=2))
for patch, p in zip(bp['boxes'], pos_order):
    patch.set_facecolor(POS_C[p]); patch.set_alpha(0.8)
ax.set(ylabel='ln(VM €)', title='Distribución del Valor de Mercado por Posición')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/G2_box_posicion.png', dpi=150, bbox_inches='tight'); plt.close()

# G3: Barras de correlación
fig, ax = plt.subplots(figsize=(10, 8))
sorted_corrs = corr_series.sort_values()
colors = ['#E74C3C' if v<0 else '#27AE60' for v in sorted_corrs.values]
ax.barh(range(len(sorted_corrs)), sorted_corrs.values, color=colors, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(sorted_corrs)))
ax.set_yticklabels(sorted_corrs.index, fontsize=8)
ax.axvline(0, color='black', lw=0.8)
ax.axvline(0.3, color='#27AE60', ls='--', lw=1, alpha=0.6, label='r=0.30')
ax.axvline(-0.3, color='#E74C3C', ls='--', lw=1, alpha=0.6)
ax.set(xlabel='r de Pearson con log(VM)', title='Correlación de Pearson: variables vs log(VM)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/G3_correlaciones.png', dpi=150, bbox_inches='tight'); plt.close()

# G4: Scatter top 6 predictores
top6 = list(corr_series.abs().nlargest(6).index)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, feat in enumerate(top6):
    ax = axes[i//3][i%3]
    sub = df_valid[[feat,'log_mv','pos_clean']].dropna()
    for pos in pos_order:
        m = sub['pos_clean']==pos
        if m.sum()>0:
            ax.scatter(sub.loc[m,feat], sub.loc[m,'log_mv'], alpha=0.3, s=12, color=POS_C[pos], label=pos)
    x,y = sub[feat].values, sub['log_mv'].values
    ok = np.isfinite(x)&np.isfinite(y)
    if ok.sum()>10:
        b = np.polyfit(x[ok],y[ok],1)
        xr = np.linspace(x[ok].min(),x[ok].max(),50)
        ax.plot(xr, np.polyval(b,xr), 'k--', lw=1.2, alpha=0.7)
    ax.set(xlabel=feat, ylabel='log(VM)' if i%3==0 else '',
           title=f'{feat}  r={corr_results[feat]["r"]:.3f}  n={corr_results[feat]["n"]}')
    if i==0: ax.legend(fontsize=6, markerscale=1.5)
plt.suptitle('Top 6 predictores vs log(VM)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/G4_scatter_top6.png', dpi=150, bbox_inches='tight'); plt.close()

# G5: PCA
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
n_show = min(12, len(eigvals))
axes[0].bar(range(1,n_show+1), exp_var[:n_show]*100, color='#3498DB', alpha=0.8)
axes[0].step(range(1,n_show+1), cum_var[:n_show]*100, where='mid', color='red', lw=2)
axes[0].axhline(80, color='orange', ls='--', lw=1, label='80%')
axes[0].axhline(90, color='green', ls='--', lw=1, label='90%')
axes[0].set(xlabel='PC', ylabel='Varianza explicada (%)', title='Scree Plot'); axes[0].legend(fontsize=9)
for pos in pos_order:
    m = pca_df['pos_clean'].values==pos
    if m.sum()>0:
        axes[1].scatter(PCs[m,0], PCs[m,1], alpha=0.3, s=12, color=POS_C[pos], label=pos)
axes[1].set(xlabel=f'PC1 ({exp_var[0]*100:.1f}%)', ylabel=f'PC2 ({exp_var[1]*100:.1f}%)',
            title='Proyección PC1 vs PC2'); axes[1].legend(fontsize=8, markerscale=2)
for j, feat in enumerate(PCA_FEATS):
    axes[2].arrow(0,0,eigvecs[j,0]*3,eigvecs[j,1]*3,
                  head_width=0.05, fc='#3498DB', ec='#3498DB', alpha=0.7)
    axes[2].text(eigvecs[j,0]*3.3, eigvecs[j,1]*3.3, feat, fontsize=7, ha='center')
circle = plt.Circle((0,0),3,fill=False,color='gray',ls='--',lw=0.8)
axes[2].add_patch(circle)
axes[2].axhline(0,color='gray',lw=0.5); axes[2].axvline(0,color='gray',lw=0.5)
axes[2].set(xlim=(-3.8,3.8), ylim=(-3.8,3.8),
            xlabel=f'PC1 ({exp_var[0]*100:.1f}%)', ylabel=f'PC2 ({exp_var[1]*100:.1f}%)',
            title='Loadings de variables', aspect='equal')
plt.suptitle('Análisis de Componentes Principales (PCA)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/G5_pca.png', dpi=150, bbox_inches='tight'); plt.close()

# G6: Edad
fig, axes = plt.subplots(1,2, figsize=(12,5))
sub = df_valid[['age','log_mv','pos_clean']].dropna()
for pos in pos_order:
    m = sub['pos_clean']==pos
    if m.sum()>5:
        axes[0].scatter(sub.loc[m,'age'], sub.loc[m,'log_mv'], alpha=0.25, s=12, color=POS_C[pos], label=pos)
x,y = sub['age'].values, sub['log_mv'].values
b2 = np.polyfit(x,y,2)
xr = np.linspace(x.min(),x.max(),100)
axes[0].plot(xr, np.polyval(b2,xr), 'k-', lw=2, label=f'Cuadrática')
axes[0].set(xlabel='Edad', ylabel='log(VM)', title='Curva de valor por edad'); axes[0].legend(fontsize=7)
axes[1].hist(df_valid['age'].dropna(), bins=30, color='#9B59B6', edgecolor='white', lw=0.4, alpha=0.85)
axes[1].axvline(df_valid['age'].median(), color='red', ls='--', lw=1.5)
axes[1].set(xlabel='Edad', ylabel='Frecuencia', title='Distribución de edades')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/G6_edad.png', dpi=150, bbox_inches='tight'); plt.close()

print('  6 gráficos guardados')

Generando gráficos...
  6 gráficos guardados


## Construcción del Excel auditado

In [13]:
print('Construyendo Excel auditado...')
wb = Workbook()
wb.remove(wb.active)

# ── HOJA 0: DATOS BRUTOS ──
ws_data = wb.create_sheet('Datos_Brutos')

EXPORT_COLS = ['name','league','pos_clean','age','min90',
               'goals','shots','shots_on_target','shot_accuracy','shots_p90',
               'tackles_won','interceptions','fouls','fouled','crosses',
               'prog_carries','prog_passes','prog_rec',
               'contract_years','club_revenue',
               'total_injuries','avg_inj_days','avg_inj_count','inj_ewa','inj_last_season',
               'xg','xa','career_maturity',
               'gk_saves','gk_save_pct','gk_clean_sheets','gk_ga90',
               'mv_eur','log_mv']
EXPORT_COLS = [c for c in EXPORT_COLS if c in df_valid.columns]
df_export = df_valid[EXPORT_COLS].copy().reset_index(drop=True)

ws_data.freeze_panes = 'A2'
for j, col in enumerate(EXPORT_COLS, start=1):
    hdr(ws_data, 1, j, col)
    ws_data.column_dimensions[get_column_letter(j)].width = max(12, len(col)+2)

for i, row in df_export.iterrows():
    r = i + 2
    for j, col in enumerate(EXPORT_COLS, start=1):
        v = row[col]
        if pd.isna(v): v = None
        fill = ST_FILL if r % 2 == 0 else WH_FILL
        c = ws_data.cell(row=r, column=j, value=v)
        c.fill, c.font, c.border, c.alignment = fill, N_FONT, BORDER, C_ALIGN
        if col == 'mv_eur':    c.number_format = '#,##0'
        elif col == 'log_mv':  c.number_format = '0.0000'
        elif col == 'club_revenue': c.number_format = '#,##0'
        elif col not in ('name','league','pos_clean'): c.number_format = '0.00'

DATA_ROWS = len(df_export)
print(f'  Hoja Datos_Brutos: {DATA_ROWS} filas × {len(EXPORT_COLS)} columnas')

col_letter = {col: get_column_letter(j) for j, col in enumerate(EXPORT_COLS, start=1)}
DATA_RANGE_START = 2
DATA_RANGE_END   = DATA_ROWS + 1

def col_range(col):
    return f"'Datos_Brutos'!{col_letter[col]}{DATA_RANGE_START}:{col_letter[col]}{DATA_RANGE_END}"

# ── HOJA 1: ESTADÍSTICOS UNIVARIADOS ──
ws_uv = wb.create_sheet('1. Estadísticos_UV')
ws_uv.column_dimensions['A'].width = 34
for c in 'BCDEFGH': ws_uv.column_dimensions[c].width = 16

ws_uv.merge_cells('A1:G1')
c = ws_uv['A1']
c.value = 'Estadísticos Univariados — TODAS las celdas son fórmulas Excel sobre Datos_Brutos'
c.fill, c.font, c.alignment = H_FILL, Font(color='FFFFFF', bold=True, size=12), C_ALIGN

row = 4
for j, label in enumerate(['Variable','N (no vacíos)','Media','Mediana','Desv. Típica','Mínimo','Máximo','Fórmula usada'], start=1):
    hdr(ws_uv, row, j, label)
ws_uv.column_dimensions['H'].width = 32

row = 5
STAT_FEATURES = [c for c in EXPORT_COLS if c not in ('name','league','pos_clean')]

for feat in STAT_FEATURES:
    cl = col_letter.get(feat)
    if not cl: continue
    rng = f"'Datos_Brutos'!{cl}{DATA_RANGE_START}:{cl}{DATA_RANGE_END}"
    stripe = WH_FILL if row % 2 == 0 else ST_FILL
    val(ws_uv, row, 1, feat, fill=stripe, bold=True).alignment = L_ALIGN
    formula(ws_uv, row, 2, f'=COUNTA({rng})-COUNTBLANK({rng})', fill=stripe, fmt='#,##0')
    formula(ws_uv, row, 3, f'=AVERAGE({rng})', fill=stripe, fmt='#,##0' if feat in ('mv_eur','club_revenue') else '0.0000')
    formula(ws_uv, row, 4, f'=MEDIAN({rng})', fill=stripe, fmt='#,##0' if feat in ('mv_eur','club_revenue') else '0.0000')
    formula(ws_uv, row, 5, f'=STDEV({rng})', fill=stripe, fmt='#,##0' if feat in ('mv_eur','club_revenue') else '0.0000')
    formula(ws_uv, row, 6, f'=MIN({rng})', fill=stripe, fmt='#,##0' if feat in ('mv_eur','club_revenue') else '0.0000')
    formula(ws_uv, row, 7, f'=MAX({rng})', fill=stripe, fmt='#,##0' if feat in ('mv_eur','club_revenue') else '0.0000')
    val(ws_uv, row, 8, f'=AVERAGE/MEDIAN/STDEV/MIN/MAX({col_letter[feat]}2:{col_letter[feat]}{DATA_RANGE_END})', fill=stripe).alignment = L_ALIGN
    row += 1

# ── HOJA 2: CORRELACIONES ──
ws_corr = wb.create_sheet('2. Correlaciones')
ws_corr.column_dimensions['A'].width = 30
for c in 'BCDEF': ws_corr.column_dimensions[c].width = 18

ws_corr.merge_cells('A1:F1')
c = ws_corr['A1']
c.value = 'Correlaciones con log(VM) — fórmula =CORREL(xi, log_VM)'
c.fill, c.font, c.alignment = H_FILL, Font(color='FFFFFF', bold=True, size=12), C_ALIGN

row = 4
for j, label in enumerate(['Variable','=CORREL(variable, log_VM)','N pares válidos','Cobertura %','Interpretación','Fórmula Excel'], start=1):
    hdr(ws_corr, row, j, label)
ws_corr.column_dimensions['F'].width = 50

row = 5
lmv_rng = col_range('log_mv') if 'log_mv' in col_letter else None

for feat in [f for f in STAT_FEATURES if f != 'log_mv' and f != 'mv_eur']:
    cl = col_letter.get(feat)
    if not cl or not lmv_rng: continue
    rng = col_range(feat)
    r_val = corr_results.get(feat, {}).get('r', None)
    stripe = WH_FILL if row % 2 == 0 else ST_FILL
    val(ws_corr, row, 1, feat, fill=stripe, bold=True).alignment = L_ALIGN
    f_cell = formula(ws_corr, row, 2, f'=CORREL({rng},{lmv_rng})', fill=stripe, fmt='0.0000')
    if r_val is not None and not np.isnan(r_val):
        if r_val > 0.3: f_cell.fill = GR_FILL
        elif r_val > 0.15: f_cell.fill = OR_FILL
        elif r_val < -0.15: f_cell.fill = RD_FILL
    formula(ws_corr, row, 3, f'=COUNTA({rng})-COUNTBLANK({rng})', fill=stripe, fmt='#,##0')
    formula(ws_corr, row, 4, f"=(COUNTA({rng})-COUNTBLANK({rng}))/COUNTA('Datos_Brutos'!A2:A{DATA_RANGE_END})", fill=stripe, fmt='0.0%')
    interp = ('Alta positiva' if (r_val or 0) > 0.3 else 'Moderada positiva' if (r_val or 0) > 0.15 else 'Baja/nula' if abs(r_val or 0) <= 0.15 else 'Negativa')
    val(ws_corr, row, 5, interp, fill=stripe).alignment = L_ALIGN
    val(ws_corr, row, 6, f"=CORREL({col_letter[feat]}2:{col_letter[feat]}{DATA_RANGE_END},{col_letter['log_mv']}2:{col_letter['log_mv']}{DATA_RANGE_END})", fill=stripe).alignment = L_ALIGN
    row += 1

# ── HOJA 3: PCA ──
ws_pca = wb.create_sheet('3. PCA')
ws_pca.column_dimensions['A'].width = 32
for c in 'BCDEFGHIJ': ws_pca.column_dimensions[c].width = 14

ws_pca.merge_cells('A1:J1')
c = ws_pca['A1']
c.value = 'PCA — Análisis de Componentes Principales (numpy.linalg.eigh sobre matriz de covarianza)'
c.fill, c.font, c.alignment = H_FILL, Font(color='FFFFFF', bold=True, size=12), C_ALIGN

row = 5
hdr(ws_pca, row, 1, 'Componente'); hdr(ws_pca, row, 2, 'Eigenvalue (λ)')
hdr(ws_pca, row, 3, 'Var. Explicada %'); hdr(ws_pca, row, 4, 'Var. Acumulada %')
row += 1
eigval_start_row = row
for i in range(min(len(eigvals), 15)):
    stripe = WH_FILL if row % 2 == 0 else ST_FILL
    val(ws_pca, row, 1, f'PC{i+1}', fill=stripe)
    val(ws_pca, row, 2, round(float(eigvals[i]), 6), fill=stripe, fmt='0.000000')
    val(ws_pca, row, 3, round(float(exp_var[i]*100), 4), fill=stripe, fmt='0.0000')
    val(ws_pca, row, 4, round(float(cum_var[i]*100), 4), fill=stripe, fmt='0.0000')
    row += 1

row += 1
hdr(ws_pca, row, 1, 'Variable (estandarizada)')
for i in range(min(5, len(eigvals))):
    hdr(ws_pca, row, i+2, f'PC{i+1}  (λ={eigvals[i]:.3f})')
row += 1
for j, feat in enumerate(PCA_FEATS):
    stripe = WH_FILL if row % 2 == 0 else ST_FILL
    val(ws_pca, row, 1, feat, fill=stripe, bold=True).alignment = L_ALIGN
    for i in range(min(5, len(eigvals))):
        loading = round(float(eigvecs[j, i]), 6)
        c = val(ws_pca, row, i+2, loading, fill=stripe, fmt='0.0000')
        if abs(loading) > 0.3:
            c.fill = GR_FILL if loading > 0 else RD_FILL
            c.font = B_FONT
    row += 1

# ── HOJA 4: OUTLIERS ──
ws_out = wb.create_sheet('4. Outliers')
ws_out.merge_cells('A1:J1')
c = ws_out['A1']
c.value = 'Detección de Outliers — Método IQR y Z-score sobre log(VM)'
c.fill, c.font, c.alignment = H_FILL, Font(color='FFFFFF', bold=True, size=12), C_ALIGN

row = 4
for j, label in enumerate(['Jugador','Liga','Posición','VM (€M)','log(VM)','Z-score','Outlier IQR','Outlier Z>3'], start=1):
    hdr(ws_out, row, j, label)
row += 1

top_out = df_valid.nlargest(25, 'mv_eur')[['name','league','pos_clean','mv_eur','log_mv','z_score','outlier_iqr','outlier_z']]
for _, r in top_out.iterrows():
    stripe = WH_FILL if row % 2 == 0 else ST_FILL
    val(ws_out, row, 1, r['name'], fill=stripe, bold=True).alignment = L_ALIGN
    val(ws_out, row, 2, r['league'], fill=stripe)
    val(ws_out, row, 3, r['pos_clean'], fill=stripe)
    val(ws_out, row, 4, round(r['mv_eur']/1e6, 1), fill=stripe, fmt='#,##0.0')
    val(ws_out, row, 5, round(r['log_mv'], 4), fill=stripe, fmt='0.0000')
    val(ws_out, row, 6, round(r['z_score'], 3), fill=stripe, fmt='0.000')
    val(ws_out, row, 7, 'SÍ' if r['outlier_iqr'] else 'No', fill=RD_FILL if r['outlier_iqr'] else stripe)
    val(ws_out, row, 8, 'SÍ' if r['outlier_z'] else 'No', fill=RD_FILL if r['outlier_z'] else stripe)
    row += 1

# ── HOJA 5: GRÁFICOS ──
ws_plt = wb.create_sheet('5. Gráficos')
ws_plt.merge_cells('A1:B1')
c = ws_plt['A1']; c.value = 'Gráficos EDA'
c.fill, c.font, c.alignment = H_FILL, Font(color='FFFFFF', bold=True, size=13), C_ALIGN

plots = [
    ('G1_distribucion_mv.png',  'A3'),
    ('G2_box_posicion.png',     'A28'),
    ('G3_correlaciones.png',    'A53'),
    ('G4_scatter_top6.png',     'A78'),
    ('G5_pca.png',              'A103'),
    ('G6_edad.png',             'A128'),
]
for fname, anchor in plots:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        img = XLImage(fpath)
        img.width = 760; img.height = 380
        ws_plt.add_image(img, anchor)

wb.save(OUT_EXCEL)
print(f'\nExcel guardado: {OUT_EXCEL}')
print('='*60)
print('EDA AUDITADO COMPLETADO')
print('='*60)
print(f'  Datos_Brutos:   {DATA_ROWS} jugadores × {len(EXPORT_COLS)} columnas')
print(f'  Correlaciones:  {len(FEATS)} variables × =CORREL(xi, log_VM)')
top3 = ', '.join(['{f}(r={r:.3f})'.format(f=f, r=corr_results[f]['r']) for f in list(corr_series.head(3).index)])
print(f'  Top 3 predictores: {top3}')

Construyendo Excel auditado...
  Hoja Datos_Brutos: 1808 filas × 19 columnas

Excel guardado: /content/EDA TFG v2.xlsx
EDA AUDITADO COMPLETADO
  Datos_Brutos:   1808 jugadores × 19 columnas
  Correlaciones:  14 variables × =CORREL(xi, log_VM)
  Top 3 predictores: club_revenue(r=0.539), contract_years(r=0.400), prog_carries(r=0.324)
